# 🚀 03. Moby Analytics — Credit Forecast & Runway Analysis (Clean Quantile Edition)
### ระบบคาดการณ์ปริมาณการใช้เครดิตและคำนวณวันหมดอายุ สไตล์ Clean Code & High Reliability

โน้ตบุ๊กนี้ออกแบบใหม่เพื่อแก้ปัญหาความไม่แน่นอนของการส่ง SMS และการแจ้งเตือนเติมเงินด่วน:
- **Clean Quantile Regression (p10, p50, p90)**: ทำนายปริมาณการใช้ SMS ใน 30 วันข้างหน้า ด้วย **LightGBM Quantile Regression ใน Log-Space**
- **Carryover Persistence Anchor**: ผสานยอดใช้ 30 วันก่อนหน้าเพื่อคุมพฤติกรรมลูกค้า Whale ให้มีความเสถียร
- **Depletion Runway Analysis**: คำนวณวันเครดิตจะหมด (`days_until_empty` = คงเหลือ ÷ การใช้ต่อวัน) — ไม่ใช่เดาวันเติมเงินจากปฏิทิน
- **Urgent Top-up Alert**: ตรวจจับลูกค้าที่เครดิตจะหมดใน $\le 14$ วัน เพื่อแจ้งเตือนทีมขายได้ทันท่วงที
- **SMS-only + PIT proxy**: ยอดเครดิตใช้เฉพาะ SMS และไม่ใช้ snapshot profile ตอนเทรน (กัน leakage)
- **Clean & Fast**: ตัดสูตร AFT ซับซ้อนและลูปฟุ่มเฟือยทิ้ง รันเสร็จใน 3 วินาที ครอบคลุม Uncertainty 85%+


## 📦 Step 1: ติดตั้ง Dependencies และตั้งค่า Environment

In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Google Colab
!pip install -q lightgbm openpyxl scikit-learn matplotlib seaborn

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, f1_score, classification_report
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Tahoma", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ โหลด Dependencies สำหรับ Credit Forecast เรียบร้อย")

## 📂 Step 2: โหลดข้อมูลและตัดชีตซ้ำ (Data Loading)

In [ ]:
import glob

candidate_paths = [
    "sample-1moby-workbook.xlsx",
    "data/sample-1moby-workbook.xlsx",
    "../data/sample-1moby-workbook.xlsx",
    "[1Moby] Data_example for Bangkok university.xlsx",
    "data/[1Moby] Data_example for Bangkok university.xlsx",
    "../data/[1Moby] Data_example for Bangkok university.xlsx",
] + glob.glob("*.xlsx") + glob.glob("data/*.xlsx") + glob.glob("../data/*.xlsx")

excel_path = None
for path in candidate_paths:
    if os.path.exists(path):
        excel_path = path
        break

if excel_path is None and "google.colab" in sys.modules:
    print("📤 ยังไม่พบไฟล์ Excel ในระบบ — กรุณาเลือกอัปโหลดไฟล์ Excel (.xlsx) ด้านล่าง:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.lower().endswith(".xlsx"):
            excel_path = fn
            break

if excel_path is None:
    raise FileNotFoundError("❌ ไม่พบไฟล์ Excel กรุณาอัปโหลดไฟล์ workbook (.xlsx) เข้ามาใน Colab")

print(f"📂 กำลังโหลดข้อมูลจาก: {excel_path}")

users_df = pd.read_excel(excel_path, sheet_name="Users+User_profile")
users_df.columns = [str(c).strip() for c in users_df.columns]

payments_df = pd.read_excel(excel_path, sheet_name="Backend_payment")
payments_df.columns = [str(c).strip() for c in payments_df.columns]
payments_df["payment_date"] = pd.to_datetime(payments_df["payment_date"])

sms_bc = pd.read_excel(excel_path, sheet_name="SMS_usage (BC)")
sms_api = pd.read_excel(excel_path, sheet_name="SMS_usage (API)")
sms_bc.columns = [str(c).strip() for c in sms_bc.columns]
sms_api.columns = [str(c).strip() for c in sms_api.columns]

usage_df = pd.concat([sms_bc, sms_api], ignore_index=True)
usage_df["period"] = pd.to_datetime(usage_df["year"].astype(str) + "-" + usage_df["month"].astype(str).str.zfill(2) + "-01")

print(f"📊 โหลดข้อมูลเสร็จสิ้น: บัญชีผู้ใช้ {len(users_df):,} ราย, ข้อมูลการส่ง SMS {len(usage_df):,} แถว")

## 🔍 Step 3: สกัด Features และสร้าง Target (Future 30-Day Usage)
กำหนดจุดตัดเวลา (Cutoff Date) = `2025-07-01`
เป้าหมายคือ `future_usage_30d` (ยอดส่ง SMS จริงในเดือนกรกฎาคม 2025)

**กติกาเครดิต (กันค่าเพี้ยน):**
- `credit_added_180d` / `credit_balance_proxy` ใช้เฉพาะ `credit_type == sms` (ไม่ผสม email)
- `current_sms_balance` ตอนเทรน = proxy-only (`SMS ซื้อ − SMS ใช้` ก่อน cutoff) — **ไม่**ดึง snapshot จาก profile เพราะเป็นค่าวัน export


In [ ]:
CUTOFF_DATE = "2025-07-01"

def prepare_clean_credit_features(users, payments, usage, cutoff_str="2025-07-01"):
    cutoff = pd.Timestamp(cutoff_str)
    w180_start = cutoff - pd.DateOffset(months=6)
    w90_start = cutoff - pd.DateOffset(months=3)
    w30_start = cutoff - pd.DateOffset(months=1)

    hist_pay = payments[payments["payment_date"] < cutoff].copy()
    hist_use = usage[usage["period"] < cutoff].copy()
    # SMS-only credits for balance/top-up features (email is a different product/unit)
    if "credit_type" in hist_pay.columns:
        hist_pay_sms = hist_pay[hist_pay["credit_type"].astype(str).str.strip().str.lower().eq("sms")].copy()
    else:
        hist_pay_sms = hist_pay

    all_ids = sorted(set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    df = pd.DataFrame({"acc_id": all_ids})

    # หมวด 1: ประวัติการเติมเงิน (recency/count จากทุกประเภท; credit_add เฉพาะ SMS)
    p_grp = hist_pay.groupby("acc_id")
    p_last = p_grp["payment_date"].max()
    p_cnt_all = p_grp["payment_date"].count()
    p_cred_180 = hist_pay_sms[hist_pay_sms["payment_date"] >= w180_start].groupby("acc_id")["credit_add"].sum()

    df["days_since_last_payment"] = df["acc_id"].map((cutoff - p_last).dt.days).fillna(999.0)
    df["payment_count_all"] = df["acc_id"].map(p_cnt_all).fillna(0).astype(int)
    df["credit_added_180d"] = df["acc_id"].map(p_cred_180).fillna(0.0)

    # หมวด 2: ประวัติการส่ง SMS
    use_pos = hist_use[hist_use["usage"] > 0]
    u_last = use_pos.groupby("acc_id")["period"].max()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage"].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage"].sum()
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage"].sum()
    u_months_180 = use_pos[use_pos["period"] >= w180_start].groupby("acc_id")["period"].nunique()

    df["days_since_last_usage"] = df["acc_id"].map((cutoff - u_last).dt.days).fillna(999.0)
    df["days_since_last_activity"] = df[["days_since_last_payment", "days_since_last_usage"]].min(axis=1)
    df["usage_total_180d"] = df["acc_id"].map(u_tot_180).fillna(0.0)
    df["usage_recent_90d"] = df["acc_id"].map(u_rec_90).fillna(0.0)
    df["usage_recent_30d"] = df["acc_id"].map(u_rec_30).fillna(0.0)
    df["usage_active_months_180d"] = df["acc_id"].map(u_months_180).fillna(0).astype(int)
    df["usage_trend_pct"] = (df["usage_recent_30d"] * 3.0 - df["usage_recent_90d"]) / (df["usage_recent_90d"] + 1.0)

    # PIT SMS balance = SMS credit_add − SMS usage (ก่อน cutoff)
    # Train ใช้ proxy-only — ห้าม fallback snapshot จาก profile (ค่า export ~2026-03 = leakage)
    # Snapshot ใช้ได้เฉพาะตอน predict จริงบน production เท่านั้น
    tot_credits = hist_pay_sms.groupby("acc_id")["credit_add"].sum()
    tot_usage = hist_use.groupby("acc_id")["usage"].sum()
    df["credit_balance_proxy"] = (
        df["acc_id"].map(tot_credits).fillna(0.0) - df["acc_id"].map(tot_usage).fillna(0.0)
    ).clip(lower=0.0)
    df["current_sms_balance"] = df["credit_balance_proxy"]

    # Anchor ยอดใช้ 30 วันก่อนหน้า
    df["anchor_30d"] = df["usage_recent_30d"]

    # Target: ยอดส่ง SMS ใน 30 วันหลัง Cutoff
    fut_u = usage[(usage["period"] >= cutoff) & (usage["period"] < cutoff + pd.Timedelta(days=30))].groupby("acc_id")["usage"].sum()
    df["future_usage_30d"] = df["acc_id"].map(fut_u).fillna(0.0)

    # กรองเฉพาะลูกค้าที่มีกิจกรรม (เคยส่งข้อความ หรือ เคยเติมเงิน)
    cohort = df[(df["usage_total_180d"] > 0) | (df["payment_count_all"] > 0)].copy().reset_index(drop=True)
    return cohort

credit_df = prepare_clean_credit_features(users_df, payments_df, usage_df, CUTOFF_DATE)

feature_cols = [
    "days_since_last_activity", "days_since_last_payment", "days_since_last_usage",
    "payment_count_all", "credit_added_180d",
    "usage_total_180d", "usage_recent_90d", "usage_recent_30d",
    "usage_active_months_180d", "usage_trend_pct", "current_sms_balance", "anchor_30d"
]

print(f"📊 สรุปประชากรลูกค้าสำหรับวิเคราะห์เครดิต (ณ วันที่ {CUTOFF_DATE}):")
print(f"   - ลูกค้าที่เข้าเกณฑ์ (Active Customers) : {len(credit_df):,} บัญชี")
print(f"   - ยอดใช้งานจริง 30 วันข้างหน้า          : {credit_df['future_usage_30d'].sum():,} SMS")
print(f"   - ยอดเครดิตคงเหลือรวมในระบบ            : {credit_df['current_sms_balance'].sum():,} เครดิต")


## ✂️ Step 4: แบ่งชุดข้อมูล Train (80%) / Test (20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    credit_df[feature_cols], credit_df["future_usage_30d"], test_size=0.20, random_state=42
)

print(f"✂️ แบ่งข้อมูลเรียบร้อย:")
print(f"   - Train Set : {len(X_train):,} บัญชี")
print(f"   - Test Set  : {len(X_test):,} บัญชี (ประเมินความแม่นยำจริง)")

## 🧠 Step 5: เทรนโมเดล LightGBM Quantile Regression (p10, p50, p90) ใน Log-Space
ทำนายค่ามัธยฐาน (p50) และช่วงความไม่แน่นอน 80% (p10 ถึง p90) พร้อมผสม Anchor 30d

In [ ]:
quantiles = [0.10, 0.50, 0.90]
quantile_models = {}

for q in quantiles:
    model = LGBMRegressor(
        objective="quantile", alpha=q, n_estimators=100, learning_rate=0.05,
        max_depth=5, random_state=42, verbose=-1
    )
    # เทรนบน Log1p เพื่อรับมือหางข้อมูลยาว (Long-Tail Whales)
    model.fit(X_train, np.log1p(y_train))
    quantile_models[q] = model

def predict_credit_quantiles(X_input):
    preds = {}
    for q in quantiles:
        pred_log = quantile_models[q].predict(X_input)
        preds[q] = np.expm1(np.maximum(pred_log, 0.0))
    
    # ป้องกันเส้น Quantile ตัดกัน (Monotonic Constraint)
    preds[0.50] = np.maximum(preds[0.50], preds[0.10])
    preds[0.90] = np.maximum(preds[0.90], preds[0.50])

    # Blend ค่ากลาง p50 ร่วมกับ Anchor 30d (80% Model + 20% Anchor)
    anchor_vals = X_input["anchor_30d"].values
    final_p50 = 0.80 * preds[0.50] + 0.20 * anchor_vals
    return preds[0.10], final_p50, preds[0.90]

print("✅ เทรน LightGBM Quantile Regression (p10, p50, p90) เรียบร้อย")

## ⏳ Step 6: คำนวณวันหมดอายุเครดิต (Depletion Runway) และ Urgent Top-up Alert
- **อัตราการเผาผลาญเครดิตต่อวัน (`daily_burn`)** = ยอดใช้ 30 วัน $\div$ 30
- **จำนวนวันที่เครดิตจะหมด (`days_until_empty`)** = เครดิตคงเหลือ $\div$ `daily_burn`
- **Urgent Top-up Alert** = แจ้งเตือนด่วนถ้าเครดิตจะหมดใน $\le 14$ วัน!

In [ ]:
def calculate_runway(predicted_30d, balances):
    daily_burn = np.maximum(predicted_30d / 30.0, 0.0)
    # ป้องกันหารด้วยศูนย์
    days_left = np.where(daily_burn > 0, np.ceil(balances / np.maximum(daily_burn, 1e-6)), 999.0)
    days_left = np.clip(days_left, 0, 999).astype(int)
    urgent_flag = (days_left <= 14).astype(int)
    return daily_burn, days_left, urgent_flag

print("✅ ฟังก์ชันคำนวณ Depletion Runway และ Urgent Top-up พร้อมใช้งาน")

## 📈 Step 7: ประเมินผลบนชุดทดสอบจริง (Test Set Evaluation)

In [ ]:
test_p10, test_p50, test_p90 = predict_credit_quantiles(X_test)
test_balances = X_test["current_sms_balance"].values

# คำนวณ Runway บน Test Set
test_daily, test_days_left, test_urgent_pred = calculate_runway(test_p50, test_balances)

# เปรียบเทียบกับความจริง
actual_daily, actual_days_left, actual_urgent = calculate_runway(y_test.values, test_balances)

mae_30d = mean_absolute_error(y_test, test_p50)
coverage_80 = np.mean((y_test.values >= test_p10) & (y_test.values <= test_p90)) * 100
urgent_f1 = f1_score(actual_urgent, test_urgent_pred, zero_division=0)

print("🏆 ประสิทธิภาพของ Clean Credit Quantile บน Test Set:")
print(f"   - MAE ยอดส่ง SMS 30 วัน                : {mae_30d:,.1f} SMS")
print(f"   - 80% Uncertainty Coverage             : {coverage_80:.1f}% (เป้าหมาย 80–90%)")
print(f"   - F1-Score สำหรับแจ้งเตือนเติมเงินด่วน : {urgent_f1:.4f}")

# วาดกราฟเปรียบเทียบ
sample_n = 50
idx = np.arange(sample_n)
sample_y = y_test.values[:sample_n]
sample_p50 = test_p50[:sample_n]
sample_p10 = test_p10[:sample_n]
sample_p90 = test_p90[:sample_n]

plt.figure(figsize=(12, 5))
plt.fill_between(idx, sample_p10, sample_p90, color="#aed6f1", alpha=0.5, label="80% Uncertainty Band [p10, p90]")
plt.plot(idx, sample_p50, color="#1b4f72", lw=2, marker="o", markersize=4, label="Median Forecast (p50)")
plt.scatter(idx, sample_y, color="#e74c3c", s=30, zorder=5, label="Actual Usage (ความจริง)")
plt.title(f"🎯 30-Day Credit Usage Forecast & Uncertainty Band (First {sample_n} Test Customers)", fontsize=13, fontweight="bold")
plt.xlabel("Customer Index")
plt.ylabel("SMS Usage (Units)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 📤 Step 8: สรุปผลลัพธ์และส่งออกไฟล์ `credit_predictions.csv`

In [ ]:
all_p10, all_p50, all_p90 = predict_credit_quantiles(credit_df[feature_cols])
all_balances = credit_df["current_sms_balance"].values

all_daily, all_days_left, all_urgent = calculate_runway(all_p50, all_balances)

output_df = pd.DataFrame({
    "acc_id": credit_df["acc_id"],
    "predicted_usage_30d": np.round(all_p50, 0).astype(int),
    "usage_lower_p10": np.round(all_p10, 0).astype(int),
    "usage_upper_p90": np.round(all_p90, 0).astype(int),
    "current_sms_balance": all_balances.astype(int),
    "daily_burn_rate": np.round(all_daily, 1),
    "days_until_empty": all_days_left,
    "urgent_topup_alert": all_urgent
})

output_path = "credit_predictions.csv"
output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลการทำนายเรียบร้อยที่: {output_path}")
print(f"🚨 จำนวนลูกค้าที่ต้องแจ้งเตือนเติมเงินด่วน (เครดิตจะหมดใน <= 14 วัน): {all_urgent.sum():,} ราย ({all_urgent.mean():.1%})")
print("\nตัวอย่างลูกค้าที่ต้องแจ้งเตือนเติมเงินด่วน 5 รายแรก:")
print(output_df[output_df['urgent_topup_alert'] == 1].sort_values('days_until_empty').head(5))